# 社区糖尿病风险筛查

## 1. 项目概述

回顾：前面已经讲了一些规范，Pipeline、网格搜索、MLflow 实验追踪 等。

完整的项目——社区糖尿病风险筛查。手里只有一个文件 `incident.db`，要交出去的是一个能给每次随访打风险分、能按 15% 名额排出复查名单的模型。

### 背景

某市社区慢病随访平台运行了三年，沉淀了三类数据：居民档案、每次随访的问卷与体测、随访期间做的化验。卫健部门希望每次随访结束后，给这位居民打一个糖尿病风险分，按分数从高到低安排糖耐量复查。每批名额大约占当批随访人数的 15%，名额之外的人排到下一批。现在这件事靠医生经验，看得准的医生一天只能看几十个人，而且标准不一。我们要把它变成一个可以批量执行、可以被检查的模型。

### 预测目标

对一次随访，预测这位居民在这次随访之后 90 天内的化验是否达到阳性标准（空腹血糖 ≥ 7.0 mmol/L，或糖化血红蛋白 ≥ 6.5%），达到为 1，否则为 0。

### 约束条件

任务书还给了两条约束：
- 模型的输入只能来自这次随访本身记录的信息（问卷、体测、病史）；
- 上线时模型只能看到这次随访的记录，随访之后的化验、下次随访的记录，都不能作为输入。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sqlite3                                              # 读 SQLite 数据库（Python 自带，不用装）
import json                                                 # 写 metrics.json、特征清单
import joblib                                               # 保存整条 Pipeline

from sklearn.model_selection import train_test_split        # 划分数据集用的
from sklearn.pipeline import Pipeline                       # 搭建流水线
from sklearn.compose import ColumnTransformer               # 数值列、类别列分开处理
from sklearn.impute import SimpleImputer                    # 填补缺失值
from sklearn.preprocessing import OneHotEncoder             # 对数据做独热编码
from sklearn.preprocessing import StandardScaler            # 对数据做标准化
from sklearn.linear_model import LogisticRegression         # 逻辑回归
from sklearn.ensemble import RandomForestClassifier         # 随机森林
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score          # 三个常用指标
from sklearn.metrics import confusion_matrix, classification_report          # 混淆矩阵、分类报告

pd.set_option("display.max_columns", 40)                    # 列多，全部显示
pd.set_option("display.width", 120)

In [2]:
SEED = 42 
DATA = "../data/"
DB = DATA + "incident.db"
# 设置后期使用的标签列的列名
TARGET = "label"

In [3]:
# with 语句: 自动打开数据库连接，代码块结束后自动关闭连接，不用手动写 conn.close()，防止忘记关闭
with sqlite3.connect(DB) as conn:
    # 从 sqlite 的系统表 sqlite_master 查询，拿到数据库里所有普通数据表的名字
    tables = pd.read_sql("select name from sqlite_master where type='table'", conn)
    print(tables)

    # 循环遍历每一张表的表名，查看表里面的信息
    for name in tables['name']:
        # PRAGMA table_info(表名)：SQLite 专属命令，查询这张表的字段信息（列名、类型、是否主键等）
        cols = pd.read_sql(f"PRAGMA table_info({name})",conn)
        # COUNT(*) 统计表总行数；取出结果里n这一列第0个值，就是行数
        n = pd.read_sql(f"select count(1) as n from {name}",conn)["n"][0]
        # 格式化打印输出：表名左对齐占12字符、行数右对齐、列数量，最后拼接所有列名
        print(f"{name:<12}{n:>8}行 {len(cols):>2}列: ", ", ".join(cols["name"]))

          name
0    residents
1       visits
2  lab_results
residents      36000行  7列:  resident_id, gender, birth_date, education, income_level, district, enroll_date
visits        101297行 23列:  visit_id, resident_id, visit_date, height_cm, weight_kg, waist_cm, sbp, dbp, smoking_status, drinking_freq, exercise_min_week, sleep_hours, veg_fruit_daily, family_history_dm, hypertension_dx, hyperlipidemia_dx, gdm_history, polyuria, polydipsia, weight_loss_unexplained, known_diabetes, created_at, chronic_mgmt_tag
lab_results    79358行  6列:  lab_id, resident_id, sample_date, report_date, fpg_mmol_l, hba1c_pct


In [4]:
# parse_dates=[...] 在读的时候就把日期列转成 datetime，后面做日期加减不用再 pd.to_datetime
with sqlite3.connect(DB) as conn:
    residents = pd.read_sql("select * from residents",conn,parse_dates=['birth_date','enroll_date'])
    visits = pd.read_sql("select * from visits",conn,parse_dates=['visit_date','created_at'])
    labs = pd.read_sql("select * from lab_results",conn,parse_dates=['sample_date','report_date'])
    print(visits)
    print(labs)
    print(residents.shape,visits.shape,labs.shape)

         visit_id resident_id visit_date  height_cm  weight_kg  waist_cm    sbp   dbp smoking_status drinking_freq  \
0       V00000001    R0000131 2023-01-01      164.1       61.4      78.6  100.0  65.0          never      frequent   
1       V00000002    R0000342 2023-01-01      162.2       73.3      91.9  115.0  62.0          never    occasional   
2       V00000003    R0000387 2023-01-01      161.5       56.0       NaN  118.0  72.0        current         never   
3       V00000004    R0000736 2023-01-01      172.0       70.2      80.6  132.0  84.0        current    occasional   
4       V00000005    R0001057 2023-01-01      156.0       48.5      63.1  125.0  69.0          never         never   
...           ...         ...        ...        ...        ...       ...    ...   ...            ...           ...   
101292  V00101293    R0034982 2025-06-30      161.6       47.9      76.3   75.0  58.0          never    occasional   
101293  V00101294    R0035219 2025-06-30      161.5     

In [5]:
visits.head(3).T

,0,1,2
visit_id,V00000001,V00000002,V00000003
resident_id,R0000131,R0000342,R0000387
visit_date,2023-01-01 00:00:00,2023-01-01 00:00:00,2023-01-01 00:00:00
height_cm,164.1,162.2,161.5
weight_kg,61.4,73.3,56.0
waist_cm,78.6,91.9,NaN
sbp,100.0,115.0,118.0
dbp,65.0,62.0,72.0
smoking_status,never,never,current
drinking_freq,frequent,occasional,never


In [6]:
visits.describe(include="number").T[['count', 'mean', 'min', '50%', 'max']].round(1)

,count,mean,min,50%,max
height_cm,101297.0,162.3,1.4,163.0,191.3
weight_kg,101297.0,64.7,31.1,64.0,198.4
waist_cm,86036.0,81.7,39.1,81.6,123.4
sbp,101297.0,120.5,0.0,121.0,176.0
dbp,101297.0,74.5,36.0,74.0,111.0
exercise_min_week,101297.0,132.1,0.0,82.0,10080.0
sleep_hours,93165.0,6.9,3.0,6.9,11.3
veg_fruit_daily,101297.0,0.6,0.0,1.0,1.0
hypertension_dx,101297.0,0.1,0.0,0.0,1.0
hyperlipidemia_dx,101297.0,0.1,0.0,0.0,1.0


### 问题：看 min、max
身高最小 1.4（这是按"米"录的）、身高最高 191.3（这是按"厘米"录的），体重最大 198.4、收缩压最小 0、每周运动最多 10,080 分钟。**单位不统一**。

In [7]:
# 查看 visits 表中每一列的确实比例
print(visits.isna().mean().round(3).sort_values(ascending=False).head(4))

gdm_history                0.485
waist_cm                   0.151
sleep_hours                0.080
weight_loss_unexplained    0.030
dtype: float64


### 解读
- `gdm_history` 0.485：妊娠糖尿病史，近一半数据缺失（48.5%）
- `waist_cm` 0.151：腰围，15.1% 缺失
- `sleep_hours` 0.080：睡眠时长，8% 缺失

In [8]:
# `duplicated()` 判断是不是和前面一模一样的整行
print("完全重复的行：",visits.duplicated().sum())

完全重复的行： 0


In [9]:
# 根据性别统计行数
print(residents['gender'].value_counts().to_dict())

# 吸烟状态
print(visits['smoking_status'].value_counts().to_dict())

# 饮酒频率统计
print(visits["drinking_freq"].value_counts().to_dict())

# 删除完全重复的
visits = visits.drop_duplicates()
# 统一性别的类别标签，最终留下：F / M (注：底部被截断)

{'F': 18276, 'M': 17032, 'male': 186, 'female': 174, '女': 172, '男': 160}
{'never': 66495, 'current': 22022, 'former': 8615, 'never ': 961, '从不': 956, 'Never': 930, 'Current': 473, '吸烟': 471, 'Former': 203, '已戒': 171}
{'never': 49524, 'occasional': 34237, 'frequent': 14436, '不喝': 1534, '偶尔': 1097, '经常': 469}


### 问题：同一个性别，写法五花八门
英文大写简写 F/M、英文全写 male/female、中文"男/女"混在一起，属于**类别不一致**，后面建模必须统一。

### 问题：吸烟状态同样混乱
never / Never / never （末尾带空格）/ 从不 其实都是【从不吸烟】；current / Current / 吸烟 是【正在吸烟】，标签不统一。

### 问题：饮酒频率中英文混写
never 和不喝是一类、occasional 和偶尔一类、frequent 和经常一类。

## 3. 数据清洗

回顾：刚才检查到一些脏数据，现在需要进行数据的清洗，例如：统一类别的标签。

### 3.1 统一类别标签

In [10]:
# 删除完全重复的
visits = visits.drop_duplicates()

# 统一性别的类别标签，最终留下：F / M
residents['gender'] = residents['gender'].replace(
    {"male":"M","男":"M","female":"F","女":"F"}
)

# 统一吸烟状态的类别标签，最终留下：英文 never，current，former
visits['smoking_status'] = visits['smoking_status'].replace(
    {'从不':"never", 
    '吸烟':"current", 
    '已戒':'former', 
    'Never':'never', 
    'Current':'current', 
    'Former':'former', 
    'never ':'never'}
    )

# 统一饮酒频率的类别标签，最终留下：never\occasional\frequent
# 【注意】原代码末尾缺少一个右括号 `)`
visits["drinking_freq"] = visits["drinking_freq"].replace(
    {"不喝":"never", 
    "偶尔":"occasional", 
    "经常":"frequent"}
    )

print(residents['gender'].value_counts().to_dict())
print(visits['smoking_status'].value_counts().to_dict())
print(visits["drinking_freq"].value_counts().to_dict())

{'F': 18622, 'M': 17378}
{'never': 69342, 'current': 22966, 'former': 8989}
{'never': 51058, 'occasional': 35334, 'frequent': 14905}


### 3.2 单位、哨兵值、越界

这一段是数值型字段清洗，专门处理三类脏数据：
1. **单位混录**（有的人按米填身高、有的人按斤填体重）
2. **哨兵值**（用 0 代表"没测/缺失"，不是真实测量值）
3. **越界异常值**（现实里不可能出现的离谱数字）

- `sbp` 收缩压（高压），血压不可能等于 0——这里的 0 不是测量结果，是录入人员标记：这个数据缺失、没有测量，这种用来代表缺失的特殊数字，就叫**哨兵值**。处理：把 0 改成 `np.nan`，统一为 pandas 标准缺失值，方便后续缺失值处理。
- `exercise_min_week` 每周运动分钟数：3000 分钟 = 50 小时，一周总共才 168 小时，一周运动 50 小时不符合常理，属于**越界异常值**，直接标记成缺失 NaN。

In [11]:
# 按米录入的身高 → 厘米
visits.loc[visits['height_cm']<3,'height_cm'] *=100
visits[visits['height_cm']<3]

# 按斤录入的体重 → 公斤
visits.loc[visits['weight_kg'] > 150,'weight_kg'] /= 2

# sbp = 收缩压（高压），血压不可能等于 0
visits.loc[visits['sbp'] == 0,'sbp'] = np.nan

visits.loc[visits['exercise_min_week'] > 3000,'exercise_min_week'] = np.nan

visits[[
    'height_cm', 
    'weight_kg', 
    'sbp', 
    'exercise_min_week'
    ]].describe().T[
        ['min', 
        '50%', 
        'max']]

,min,50%,max
height_cm,134.4,163.0,191.3
weight_kg,31.1,64.0,149.7
sbp,62.0,121.0,176.0
exercise_min_week,0.0,82.0,2162.0


清洗后验证：身高 min 从 1.4 → 134.4（单位统一），体重 max 从 198.4 → 149.7（按斤的已转公斤），收缩压 min 从 0 → 62.0（哨兵值已处理），运动 min 从 10080 → 2162（越界异常值已处理）。

In [12]:
# 一次随访 × 这个人的每一次化验
df = visits.merge(labs,on='resident_id',how='left')
# 条件判断：窗口以内
in_window = ((df['sample_date'] >= df['visit_date']) 
& (df['sample_date']<= df['visit_date'] + pd.Timedelta(days=90)))
# drop_duplicates('visit_id',keey='first'):同一个随访，只保留窗口内最早的那一次化验
df = df[in_window].sort_values('sample_date').drop_duplicates('visit_id',keep='first')
# 空腹血糖 fpg ≥7.0 mmol/L 或 糖化血红蛋白 hba1c ≥6.5%
# `.astype(int)` → True 变成 1，False 变成 0，做成 0/1 二分类标签，给模型用
df[TARGET] = ((df['fpg_mmol_l'] >= 7.0) | (df['hba1c_pct'] >=6.5)).astype(int)
# 拼上居民档案
df = df.merge(residents,on='resident_id',how='left')
print('有标签的访问：',len(df),"阳性率：%.3f"% df[TARGET].mean())

有标签的访问： 79705 阳性率：0.299


### 3.3 打标签：随访后 90 天内的首次化验

任务：根据一次随访，看这个人在随访之后 90 天内第一次化验，判断是否糖尿病阳性。

**总结**：101,297 条随访里，有 79,705 条在窗口内找到了化验，阳性率 29.9%。在这 79705 条能打上标签的随访样本里，大约 3 成的人后续化验达到糖尿病诊断标准。

### 3.4 派生特征：年龄、BMI

年龄 = 随访日期 - 出生日期；BMI = 体重 / (身高(米))²

In [13]:
df['age'] = (df['visit_date'] - df['birth_date']).dt.days / 365.25
df['bmi'] = df['weight_kg'] / (df['height_cm'] / 100)**2
# 统计
df[['age','bmi']].describe().T[['mean','min','max']].round(1)

,mean,min,max
age,60.1,22.0,90.5
bmi,24.2,15.0,67.9


**总结**：年龄 22 到 90 岁，平均 60 岁，是社区慢病随访的典型人群；BMI 平均 24.2。

---

## 4. 选列和切分

筛选哪些列可以放进模型做特征，是建模前的一步：区分【要丢掉的列】、【数值特征】、【类别特征】。

In [14]:
# 要丢掉的列
DROP_COLS = ['visit_id','resident_id','lab_id','visit_date','created_at','birth_date',
             'enroll_date','fpg_mmol_l','hba1c_pct','sample_date','report_date',TARGET]
# 数值特征
NUM_COLS = df.select_dtypes(include="number").columns.difference(DROP_COLS).tolist()
# 类别特征
CAT_COLS = df.select_dtypes(exclude=['number','datetime']).columns.difference(DROP_COLS).tolist()
# 所有的特征
FEATURES = NUM_COLS + CAT_COLS
print('数值列',len(NUM_COLS),NUM_COLS)
print('类别列',len(CAT_COLS),CAT_COLS)

数值列 17 ['age', 'bmi', 'dbp', 'exercise_min_week', 'gdm_history', 'height_cm', 'hyperlipidemia_dx', 'hypertension_dx', 'known_diabetes', 'polydipsia', 'polyuria', 'sbp', 'sleep_hours', 'veg_fruit_daily', 'waist_cm', 'weight_kg', 'weight_loss_unexplained']
类别列 8 ['chronic_mgmt_tag', 'district', 'drinking_freq', 'education', 'family_history_dm', 'gender', 'income_level', 'smoking_status']


### 4.1 DROP_COLS = 必须扔掉、不能进模型的列

分 4 类理解：
1. **主键 ID**：`visit_id`、`resident_id`、`lab_id`，只是编号，没有预测意义，不能喂给模型。
2. **原始日期**：`visit_date`、`birth_date` 等，我们已经用它们算出了 `age` 年龄，原始日期不再使用。
3. **泄漏字段（最重要）**：`fpg_mmol_l`（空腹血糖）、`hba1c_pct`（糖化血红蛋白）——我们的标签 `TARGET` 就是靠这两个化验值算出来的！如果把这两列放进模型，相当于直接把答案给模型，属于**数据泄露**，模型会假的特别准，上线完全没用。
4. **TARGET**：标签本身，是我们要预测的目标，当然不能当做特征输入。

### 4.2 数值特征

`select_dtypes(include="number")`：自动选出所有数字类型列，`.difference(DROP_COLS)`：去掉上面 DROP 列表里要丢弃的列。

### 4.3 类别特征

`exclude=["number", "datetime"]`：排除数字、日期，剩下的就是文本类别列。

### 4.4 划分训练集、验证集，落盘

In [15]:
# 划分
train,valid = train_test_split(df,test_size=0.3,random_state=SEED,stratify=df[TARGET])
# 把划分好的训练集、验证集保存成 parquet 文件，
# 后面训练模型的时候直接读取，不用反复跑前面清洗、特征工程代码
# index=False: 不保存行索引
train.to_parquet('train.parquet',index=False)
valid.to_parquet('valid.parquet',index=False)
# 分开特征 X 和 标签 y
X_train,y_train = train[FEATURES],train[TARGET]
X_valid,y_valid = valid[FEATURES],valid[TARGET]
print(train.shape,valid.shape)
print("阳性率 训练 %.3f 验证 %.3f" % (y_train.mean(),y_valid.mean()))

(55793, 37) (23912, 37)
阳性率 训练 0.299 验证 0.299


## 5. 建模、评估与交付

主要：sklearn Pipeline 流水线，把【缺失值填充、标准化、独热编码、逻辑回归】打包成一整条链路，是建模的标准写法。

### 5.1 流水线：预处理 + 逻辑回归

In [16]:
# 数值列：中位数填补 + 标准化；类别列：众数填补 + 独热。每个模型各造一份，不共用
def make_pre(num_col,cat_col):
    return ColumnTransformer([
        ('num',Pipeline([
            # 缺失值用中位数填充
            ('impute',SimpleImputer(strategy='median')),
            # 标准化
            ('scale',StandardScaler())
        ]),num_col),
        ('cat',Pipeline([
            # 缺失值用众数填充
            ('impute',SimpleImputer(strategy='most_frequent')),
            # 独热编码
            ('onehot',OneHotEncoder(handle_unknown='ignore'))
        ]),cat_col)
    ])
# 整条流水线
pipe = Pipeline([
    # 预处理
    ('pre',make_pre(NUM_COLS,CAT_COLS)),
    # 逻辑回归模型
    ('lr',LogisticRegression(max_iter=2000))
])
# 训练
pipe.fit(X_train,y_train)
print('预处理之后的列数：',pipe.named_steps['pre'].transform(X_train.head(1)).shape[1])

预处理之后的列数： 47


**原始特征**：17 数值 + 8 类别 = 25 列。17 个数值列：做完中位数填充 + 标准化，数量不变，还是 17 列；8 个类别列：OneHot 独热展开，变成 47 - 17 = 30 列。

### 5.2 评估逻辑回归

In [17]:
# predict_proba 输出两列：[阴性概率, 阳性概率]，[:,1] 取出【是糖尿病阳性的概率】，取值 0~1
pro = pipe.predict_proba(X_valid)[:,1]
# 阈值 0.5：概率 ≥0.5 → 预测为阳性 1；小于 0.5 → 阴性 0
pred = (pro>=0.5).astype(int)
print({
    # 准确率
    "accuracy":round(accuracy_score(y_valid,pred),4),
    # F1 分数
    "f1": round(f1_score(y_valid,pred),4),
    # AUC 值
    "roc_auc":round(roc_auc_score(y_valid,pred),4)
})
# 混淆矩阵
print(confusion_matrix(y_valid,pred))

{'accuracy': 0.9558, 'f1': 0.9244, 'roc_auc': 0.9406}
[[16400   361]
 [  695  6456]]


**指标解读**：
- `accuracy = 0.9558`（准确率 95.58%）：所有样本里预测对的比例。这个数据集阳性率只有 29.9%，单纯看准确率容易骗人，只做参考。
- `f1 = 0.9244`（F1=92.44%）：精确率和召回率的调和平均，兼顾"不误报"和"不漏检"，是不平衡分类很重要的指标，这个数值很不错。
- `roc_auc = 0.9406`（AUC=0.9406）：AUC 衡量模型区分正负样本的能力，范围 0~1，越接近 1 越好。0.9406 说明这个逻辑回归模型区分糖尿病人和非糖尿病人的能力很强，效果优秀。

**混淆矩阵格式**：`[[TN, FP], [FN, TP]]`
- TN=16400：真实阴性，模型预测阴性（正确识别健康人）
- FP=361：真实阴性，模型预测阳性（误报：健康人被错判成高危）
- FN=695：真实阳性，模型预测阴性（漏检：糖尿病人没被找出来）
- TP=6456：真实阳性，模型预测阳性（正确抓到糖尿病人）

### 5.3 流水线：预处理 + 随机森林

In [18]:
# 整条流水线
pipe_rf = Pipeline([
    # 预处理
    ('pre',make_pre(NUM_COLS,CAT_COLS)),
    # 随机森林模型
    ('rf',RandomForestClassifier(n_estimators=200,random_state=SEED,n_jobs=1))
])
# 训练
pipe_rf.fit(X_train,y_train)
# predict_proba 输出两列：[阴性概率, 阳性概率]，[:,1] 取出【是糖尿病阳性的概率】，取值 0~1
pro_rf = pipe_rf.predict_proba(X_valid)[:,1]
# 阈值 0.5：概率 ≥0.5 → 预测为阳性 1；小于 0.5 → 阴性 0
pred_rf = (pro_rf>=0.5).astype(int)
print({
    # 准确率
    "accuracy":round(accuracy_score(y_valid,pred_rf),4),
    # F1 分数
    "f1": round(f1_score(y_valid,pred_rf),4),
    # AUC 值
    "roc_auc":round(roc_auc_score(y_valid,pred_rf),4)
})
# 打印报告
print(classification_report(y_valid,pred_rf,digits=3))

{'accuracy': 0.9596, 'f1': 0.9312, 'roc_auc': 0.9468}
              precision    recall  f1-score   support

           0      0.964     0.979     0.971     16761
           1      0.948     0.915     0.931      7151

    accuracy                          0.960     23912
   macro avg      0.956     0.947     0.951     23912
weighted avg      0.959     0.960     0.959     23912



**总结**：逻辑回归 accuracy=0.9558，f1=0.9244，auc=0.9406；随机森林 accuracy=0.9596，f1=0.9312，auc=0.9468。**结论**：随机森林整体效果略好于逻辑回归。

### 5.4 两个模型放一起，挑 AUC 高的

In [19]:
results = {
    "lr":{
        # 准确率
        "accuracy":round(accuracy_score(y_valid,pred),4),
        # F1 分数
        "f1": round(f1_score(y_valid,pred),4),
        # AUC 值
        "roc_auc":round(roc_auc_score(y_valid,pred),4) 
    },
    "rf":{
        # 准确率
        "accuracy":round(accuracy_score(y_valid,pred_rf),4),
        # F1 分数
        "f1": round(f1_score(y_valid,pred_rf),4),
        # AUC 值
        "roc_auc":round(roc_auc_score(y_valid,pred_rf),4)  
    }
}
best = max(results,key=lambda k:results[k]['roc_auc'])
best_pipe = {'lr':pipe,'rf':pipe_rf}[best]
pd.DataFrame(results).T

,accuracy,f1,roc_auc
lr,0.9558,0.9244,0.9406
rf,0.9596,0.9312,0.9468


In [20]:
# 复盘 1：筛选的对象是谁？
# 任务书说"给居民打风险分，安排糖尿病复查"。已经确诊糖尿病的人，还需要被筛查吗？
# 考虑问题：我们筛查的目标人群到底是谁？已确诊糖尿病的人，该不该放进模型数据集？
# 目前主要关注的字段：known_diabetes：0=未确诊的居民  1=已确诊的居民

# 统计 known_diabetes 字段各个取值的数量：0=之前没有确诊糖尿病，1=已经确诊糖尿病
print(visits['known_diabetes'].value_counts().to_dict())

# 按照【是否已确诊糖尿病】分组，统计每组样本数量size、本组内标签TARGET的均值（也就是阳性率），保留3位小数
print(df.groupby('known_diabetes')[TARGET].agg(['size','mean']).round(3))

# 筛选出【未确诊人群 known_diabetes == 0】，计算这部分人群的TARGET均值（阳性率）并打印
print("剔除已确诊之后，阳性率 %.3f" % df.loc[df['known_diabetes']==0,TARGET].mean())

{0: 84029, 1: 17268}
                 size   mean
known_diabetes              
0               64577  0.143
1               15128  0.965
剔除已确诊之后，阳性率 0.143


### 复盘1 总结：
1. 看原始数据里，有多少人是之前没有确诊、多少人已经确诊糖尿病。输出 `{0: 84029, 1: 17268}`，说明数据里混了大量已经确诊的患者。
2. 分组统计：
   - `known_diabetes=0`（未确诊居民）：样本 64577，阳性率只有 **0.143**；
   - `known_diabetes=1`（已经确诊的人）：样本 15128，阳性率高达 **0.965**；
   - 已经确诊的这批人，几乎标签都是阳性，是天然的"高分样本"。如果不剔除，模型很容易靠这个特征刷出很高 AUC，造成虚高。
3. 单独看真正需要筛查的人群（未确诊）的阳性率：**0.143**。这个 0.143 才是我们这个糖尿病筛查任务真实的基线分布，后面建模、评估指标，都应该基于这个人群，而不是带上已确诊病人。

**结论**：任务目标是给居民打分、安排糖耐复查，已经确诊糖尿病的人不属于本次筛查对象，必须从训练、测试集剔除；如果不剔除，模型学到的不是"谁会新发糖尿病"，而是识别"这个人本来就有病"，指标再好也不符合业务需求。

In [21]:
# 复盘 2：chronic_mgmt_tag 在随访那天存在吗？
# 问题：chronic_mgmt_tag（慢病纳管标签）看起来和别的列一样，取值只有四种。它能不能当特征？
# 校验这个字段的时间因果问题（特征泄露），是机器学习里非常经典的因果校验：
# 特征必须是【随访当时就已经存在、在结局发生之前就有的信息】，不能是结局发生之后才产生的标签。

# 筛选出还没有确诊糖尿病的人群（本次筛查目标人群）
screen = df[df['known_diabetes']==0]
# 按慢病纳管标签chronic_mgmt_tag分组，统计每组样本数量size、标签TARGET均值（阳性率），保留3位小数
print(screen.groupby('chronic_mgmt_tag')[TARGET].agg(['size','mean']).round(3))

# 按居民id分组，统计每个居民拥有的不同慢病标签的数量
n_tag = visits.groupby('resident_id')['chronic_mgmt_tag'].nunique()
# 统计【每个居民标签种类数】的分布，转为字典打印
print("每个居民有几种tag:",n_tag.value_counts().to_dict())

# 提取居民R0000014的多条随访记录：随访日期、是否既往确诊糖尿病、慢病纳管标签，用来时序观察
visits.loc[visits['resident_id']=='R0000014',
['visit_date','known_diabetes','chronic_mgmt_tag']]

                   size   mean
chronic_mgmt_tag              
both                916  0.940
diabetes           4668  0.929
hypertension       1357  0.333
none              57636  0.062
每个居民有几种tag: {1: 36000}


,visit_date,known_diabetes,chronic_mgmt_tag
2081,2023-01-22,0,diabetes
23332,2023-08-13,1,diabetes
47214,2024-02-20,1,diabetes
68451,2024-08-26,1,diabetes


### 复盘2 总结：
1. 在未确诊人群里，不同慢病标签的阳性率差异巨大：
   - `both`（同时糖尿病+高血压纳管）**0.940**
   - `diabetes`（糖尿病纳管）**0.929**
   - `hypertension`（高血压纳管）**0.333**
   - `none`（无慢病纳管）**0.062**
2. 输出 `{1:36000}`：每个居民全程只有 **1 种固定 tag**，标签不会随随访时间变化。
3. 拿单个居民 `R0000014` 的时序记录是关键证据：
   - 2023-01-22：`known_diabetes=0`（当时还没确诊），但是标签已经是 `diabetes`
   - 后面几次随访：`known_diabetes=1`（已经确诊糖尿病），标签依旧是 `diabetes`
   - `chronic_mgmt_tag` 这个糖尿病纳管标签，是在这个人后续确诊糖尿病之后，才给打上的慢病管理标签，是**结果**，不是事前的基线特征。
   - 虽然在第一条随访记录里它也显示为 `diabetes`，但这个标签本质是慢病系统的纳管标识，是结局相关的后置标签，不是随访当天已经存在的前置风险因素。

**结论**：如果直接拿 `chronic_mgmt_tag` 当特征，属于**特征泄露（data leakage）**：模型不是学习风险因素，而是直接拿到了事后的慢病登记结果，指标虚高，上线之后无法使用。`chronic_mgmt_tag` 不能直接拿来做本次筛查模型的特征——这个标签是慢病管理的登记标签，和本次是否新发糖尿病这个结局强耦合，存在时间上的因果倒置，属于泄露特征。

In [22]:
# 复盘 3：考卷是哪一年的——时间切分校验
# 问题：上线以后，模型每天面对的是"明天来随访的人"。我们的验证集，能代表明天的人吗？
# 模型上线后，预测的是"明天新来随访的居民"，随机切分得到的验证集，能不能真实模拟上线后的场景？
# 这一段是校验时间泄露、样本切分方式是否合理，是时序类风险模型非常关键的校验。

# 交叉表：统计每一年的随访数据，有多少划分到train训练集、多少划分到valid验证集
print(pd.crosstab(df['visit_date'].dt.year,np.where(df.index.isin(train.index),'train','valid')))

# 将训练集 和 验证集的居民ID、随访日期合并，同一个居民的训练随访、验证随访放到同一行
later = train.merge(valid[['resident_id','visit_date']],on='resident_id',suffixes=("","_v"))
# 计算：验证集中的居民，曾经在训练集里出现过的占比
print("验证集里的居民，在训练集里出现过的比例 %.3f" % valid["resident_id"].isin(train["resident_id"]).mean())
# 统计：同一个居民，训练集里的随访日期 > 验证集随访日期 的记录条数（训练数据用了验证时间之后的数据）
print("训练集里比同一居民的验证随访更晚的随访 %d 条" % (later["visit_date"] > later["visit_date_v"]).sum())

col_0       train  valid
visit_date              
2023        23699  10334
2024        22868   9672
2025         9226   3906
验证集里的居民，在训练集里出现过的比例 0.761
训练集里比同一居民的验证随访更晚的随访 15227 条


### 复盘3 总结：
1. 可以看到 2023、2024、2025 每一年的数据，都同时分到了训练集和验证集——普通随机切分，是打乱时间，按样本随机抽签划分，不是按时间先后切分。
2. 第二行合并居民随访记录：把同一个居民的训练随访、验证随访配对，方便对比两者的时间先后。
3. 第三行输出：验证集居民在训练集出现比例 **0.761**——意思：76.1% 的验证居民，之前已经在训练集里见过这个居民了。模型不是"第一次见这个居民"，它已经学习过这个人的历史信息，和上线场景矛盾：上线时来的人，模型应该是第一次对他做风险打分。
4. 第四行输出：训练集里比同一居民验证随访更晚的随访有 **15227 条**——这是最严重的时间泄露：验证随访是更早的时间点，训练集却拿了这个居民之后更晚的随访数据去训练。相当于考试的时候，考卷是过去的题，但是复习资料里包含了这道题之后的答案，模型作弊。

**结论**：随机切分的验证集不能代表上线后"明天来随访的人"。上线场景：时间往前走，只能用【过去的历史数据】训练，去预测【未来新的随访】；随机切分的问题：同一个人，未来的随访数据跑到训练集，验证集里很多居民已经在训练集见过。

**正确做法**：时间切分（时间回溯验证），用早于某个时间点的数据训练，用该时间点之后的数据做验证，模拟真实上线。随机切分让模型用未来训练、在过去考试。真实上线时，模型只见过过去，要预测的是未来——随机切分的验证分数回答不了"上线以后会怎样"。同一个居民出现在训练和验证两边是允许的（同一社区的人会反复来随访），禁止的是同一次随访跨集。

In [23]:
# 5.5 出名单：15%名额
# 任务书要的不是 0/1，是一张名单：每批随访按风险分排序，前 15% 去复查。名单里有多少是真阳性？

# 给验证集valid新增一列score，score是最优模型预测的糖尿病阳性风险概率（0~1）
valid = valid.assign(score = best_pipe.predict_proba(X_valid)[:,1])
# 计算名额：总验证人数 × 15%，转为整数，也就是本次筛选最多选出15%的人复查
k = int(round(len(valid) * 0.15))
# 按score风险概率从大到小，取出风险最高的k个人，作为优先复查名单
top = valid.nlargest(k,"score")
# 打印：名额总人数、这份名单里真正是糖尿病阳性的人数
# 主要通过 label标签进行判别，是不是真的阳性
print("名额",k,"人：名单里阳性",top[TARGET].sum(),"人")
# 打印两个指标：名单内阳性占比、这份名单抓到了全部阳性病人的比例
print("名单阳性占比 %.3f；抓住全部阳性的 %.3f"
     %(top[TARGET].mean(),top[TARGET].sum()/valid[TARGET].sum()))
# 取出需要交付的字段：就诊id、居民id、就诊时间、风险分数，只展示前5行预览
top[['visit_id','resident_id','visit_date','score']].head()

名额 3587 人：名单里阳性 3474 人
名单阳性占比 0.968；抓住全部阳性的 0.486


,visit_id,resident_id,visit_date,score
18994,V00025726,R0000940,2023-09-01,1.0
36644,V00046986,R0027521,2024-02-17,1.0
7930,V00009804,R0018693,2023-04-09,1.0
36207,V00046498,R0031696,2024-02-12,1.0
34681,V00042734,R0000664,2024-01-01,1.0


In [24]:
# 5.6 保存模型、特征清单、指标
# 使用joblib保存最优的整条流水线Pipeline（包含预处理+随机森林模型）到model.pkl文件
# 保存最优的模型
joblib.dump(best_pipe,'model.pkl')
# 把特征名称列表FEATURES保存到feature_cols.json，utf-8编码，中文不乱码
json.dump(FEATURES,open('feature_cols.json','w',encoding='utf-8'),ensure_ascii=False,indent=2)
# 把逻辑回归、随机森林两个模型的评估指标results保存到metrics.json
# 作用：留存模型对比记录，方便写报告、版本追溯
json.dump(results,open('metrics.json','w',encoding='utf-8'),ensure_ascii=False,indent=2)
# 将筛选出来的15%高危复查名单，保存为csv文件，不写入pandas自带行索引
# 这个csv就是交付给业务/疾控的复查名单
top[['visit_id','resident_id','visit_date','score']].to_csv('top15.csv',index=False)
# 打印选出的最优模型标识，rf代表随机森林
print('best',best)

best rf


---

## 交付物

现在的状态：

| 文件 | 说明 |
|---|---|
| `train.parquet` / `valid.parquet` | 55,793 / 23,912 行，37 列，随机切分，阳性率都是 29.9% |
| `model.pkl` | 随机森林，整条 Pipeline，可以直接加载 |
| `feature_cols.json` | 25 列：17 个数值、8 个类别 |
| `metrics.json` | 逻辑回归 AUC 0.9406，随机森林 AUC 0.9468，准确率 96% |
| `top15.csv` | 3,587 人，名单里阳性占 96.8% |
| `predict.py` | 一条记录进，一个风险分出 |

### 5.8 扩展：使用AI，如何提示AI 模型

到目前位置，生成的文件。

【提示词】我有一个 SQLite 数据库 `incident.db`，里面有三张表：`residents`（居民档案）、`visits`（随访记录）、`lab_results`（化验结果）。请帮我写一个 Python 脚本：
1. 读取三张表，做数据清洗（缺失值、异常值、重复值、类别不统一）；
2. 构造标签：随访后 90 天内的化验中，空腹血糖 >= 7.0 或糖化血红蛋白 >= 6.5 记为 1，否则为 0；
3. 做特征工程（年龄、BMI 等），类别变量独热编码；
4. 划分训练集和验证集，保存成 `train.parquet` 和 `valid.parquet`；
5. 训练逻辑回归和随机森林，输出准确率、F1、AUC、混淆矩阵和分类报告；
6. 保存最好的模型，并写一个预测函数。

In [25]:
# 计算：采样日期到报告日期相差多少天，也就是报告延迟时长
lag = (labs['report_date'] - labs['sample_date']).dt.days
# 打印延迟时长的统计量：最小值、中位数、最大值，看报告多久能出来
print(lag.describe()[['min','50%','max']].to_dict())

# 定义标签生成函数：给随访记录打标签
# 参数v：随访表；as_of：快照截止日（站在这一天，只看这天之前已经出的报告）
# sort_by：按哪个字段排序，默认按采样时间sample_date
def label_visits(v, as_of, sort_by='sample_date'):
    # 筛选：在快照日as_of之前，报告已经生成的化验记录
    known = labs[labs['report_date'] <= as_of]
    # 随访表和已出报告的化验表，按居民id关联，拿到同一个人的随访和化验
    m = v[[
        'visit_id', 
        'resident_id', 
        'visit_date'
        ]].merge(known,on='resident_id')
    # 两个筛选条件：
    # 1. 化验采样时间 >= 随访当天（是随访之后抽的血，不是随访之前旧的化验）
    # 2. 采样时间 <= 随访日期 +90 天（90天观察窗口，只看随访后90天内的化验）
    m = m[(m['sample_date'] >= m['visit_date']) & (m['sample_date'] <= m['visit_date'] + pd.Timedelta(days=90))]
    # 按采样时间排序，每个随访visit_id只保留第一条（首次化验）
    first = m.sort_values(sort_by).drop_duplicates('visit_id')
    # 打标签TARGET：空腹血糖>7.0 或者 糖化血红蛋白≥6.5，满足任意一个就是阳性，转为0/1整数
    first[TARGET] = ((first['fpg_mmol_l'] >= 7.0) | (first['hba1c_pct']>=6.5)).astype(int)
    # 把标签合并回原始随访表，返回带标签的随访数据
    return v.merge(first[['visit_id',TARGET]],on='visit_id')

# 筛选随访样本：
# 条件1：known_diabetes ==0，也就是之前没有确诊糖尿病的人
# 条件2：随访时间在2024-03-03 ~ 2024-06-30之间，这批是我们用来做标签的随访（快照日前4个月的随访）
spring = visits[(visits['known_diabetes']==0) & (visits['visit_date'].between('2024-03-03','2024-06-30'))]

# 循环两个不同快照时间点，对比看标签变化
for as_of in ['2024-06-30', '2025-06-30']:
    # 调用打标签函数，站在 as_of 这个时间快照，生成标签
    lab_ = label_visits(spring, pd.Timestamp(as_of))
    # 输出：站在这个快照日，一共有多少条拿到标签、阳性样本一共多少条
    print(f"站在{as_of}看：有标签{len(lab_):>5}条，阳性 {lab_[TARGET].sum():>4}条")

{'min': 1.0, '50%': 6.0, 'max': 20.0}
站在2024-06-30看：有标签 7270条，阳性 1048条
站在2025-06-30看：有标签 8780条，阳性 1242条


### 复盘4：标签什么时候才算"有了"

问题：按时间切的话，训练集截到哪一天？截到"今天"行不行？

这要预测：一个新随访的人，会不会在随访之后 90 天内查出糖尿病。

**总结**：

（1）能不能直接把训练集截止到"今天"？
答案：**不行！** 如果快照日选今天，很多随访已经在 90 天观察窗口内抽了血，但是报告还没生成、还没入库。这部分样本，模型会误以为它是阴性（没患病），但实际报告晚几天出来之后，它是阳性 → 标签被低估，样本标签错误，模型学歪。

（2）代码里做的验证实验：同一批随访（spring：2024-03-03 ~ 2024-06-30 的随访）
1. 站在 2024-06-30（早快照）：很多报告还没出，只有 7270 条拿到标签，阳性 1048
2. 站在 2025-06-30（晚快照，等了很久，所有报告都已经入库）：拿到标签涨到 8780 条，阳性 1242

**结论**：快照时间越晚，等待足够长的报告缓冲期，能拿到的标签越多、阳性样本才完整，标签才真实。

In [26]:
# 筛选：既往没有确诊糖尿病的随访记录，作为本次筛查人群
screen_visits = visits[visits['known_diabetes']==0]

# 判断：这条随访id，不在已经拿到化验标签的df里面 -> 也就是随访90天窗口内没有任何化验
no_lab = ~screen_visits['visit_id'].isin(df['visit_id'])

# 打印总筛查随访数量，以及窗口内没有化验的随访条数
print('未确诊的随访', len(screen_visits), "条，其中窗口内没有化验", no_lab.sum(), '条')

# 只看【有化验】的人群，计算阳性率：阳性样本 / 有化验的总人数
print('只算有化验的：阳性率 %.3f' % screen[TARGET].mean())

# 错误做法：把【没有化验的人直接当成阴性】，用阳性总数 / 全部随访人群（含没抽血的）
print("没有化验当成阴性：阳性率 %.3f" % (screen[TARGET].sum() / len(screen_visits)))

未确诊的随访 84029 条，其中窗口内没有化验 19452 条
只算有化验的：阳性率 0.143
没有化验当成阴性：阳性率 0.110


### 复盘5：没化验 ≠ 阴性

标签规则：随访后 90 天内抽血化验，血糖达标才标记阳性。

**误区**：一个人在观察窗口里没有抽血化验，不等于他没有糖尿病（不等于阴性），只是我们没有拿到他的化验结果，他的标签是缺失、未知，**不是 0**。

**总结**：

（1）两组对照，看两种处理方式带来的阳性率差别
1. 只统计有化验的人：阳性率 **0.143**（14.3%）
2. 错误做法：把没化验的人全部当成阴性，强行参与计算：阳性率降到 **0.110**（11.0%）

**结论**：如果直接 `fillna(0)`，把无化验的样本填充为阴性，会压低整体阳性率，标签分布失真，模型训练会出问题。

**正确做法**：窗口内没有已出报告的化验，这条随访不进数据集——它不是阴性，是不知道。

In [27]:
# 打印原始样本总数，以及对特征集直接执行dropna删除缺失行之后剩下的样本数量
print(len(screen), "->", len(screen[FEATURES].dropna()))

# 对dropna之后剩下的数据，统计gender（性别）分布，转为字典输出
print(screen[FEATURES].dropna()['gender'].value_counts().to_dict())

# 按性别分组，计算每组gdm_history（妊娠史）的缺失比例，保留3位小数，转为字典
print(screen.groupby('gender')['gdm_history']
    .apply(lambda s:s.isna().mean())
    .round(3).to_dict())

64577 -> 19731
{'F': 19731}
{'F': 0.0, 'M': 1.0}


### 复盘6：dropna 删掉了谁

**总结**：

（1）第一行：原始一共 64577 条样本，直接 `dropna()` 删除任意特征有缺失的行之后，只剩 19731 条，大量样本被删掉。
（2）第二行：删完之后，剩下的全部都是女性 F，男性样本一条都不剩了。
（3）第三行：分组看妊娠史缺失率：女性的妊娠史 `gdm_history` 没有缺失（0.0）；男性的妊娠史全部是缺失（1.0，100% 缺失）。

**核心知识点**：缺失不一定是随机缺失（MCAR），很多是业务上的**结构性缺失**。

- `gdm_history` 妊娠史这个字段，业务上男性本来就没有这个指标，不是"漏录了"，是这个字段对男性本身就无意义，天然为空。
- 如果直接无脑写 `dropna()`：只要任意一个特征存在缺失，整行删掉。男性因为妊娠史字段天然缺失，会被全部删除。
- 后果：样本分布被悄悄改变，原本男女都有的数据集，直接只剩女性，样本有偏，模型完全不能用到男性人群上。

这段代码用来演示：盲目 `dropna` 非常危险，结构性缺失会把特定人群直接删掉，样本分布发生偏移；缺失要先区分是业务上的合理缺失，还是随机漏录。

**正确做法**：数据集里保留缺失；填补放进 Pipeline，只在训练集上拟合，并加缺失指示列。

In [28]:
# 打开sqlite数据库连接，自动管理连接，结束后自动关闭
with sqlite3.connect(DB) as conn:
    # 读取随访表：居民ID、随访日期、体重kg；同时把随访日期自动解析成时间格式
    raw = pd.read_sql("select resident_id,visit_date,weight_kg from visits",conn,parse_dates=['visit_date'])

# 按照居民id、随访日期排序，保证同一个人的随访记录按时间先后排列
raw.sort_values(['resident_id','visit_date'])
# 按居民分组，对体重字段做计算
# shift()：取上一条记录（也就是本次之前的历史数据）
# expanding()：滚动扩展，只使用【当前时刻之前】的所有历史数据
# median()：求前面所有历史体重的中位数
# transform：计算结果和原表行数对齐，返回给每一行
prior = (raw.groupby('resident_id')['weight_kg']
         .transform(lambda s: s.shift().expanding().median()))

# 统计：体重高于150kg的记录一共有多少条
print(">150kg的条数",(raw['weight_kg']>150).sum())

# 统计：当前体重 ÷ 历史中位数，落在1.8~2.2之间的记录条数
# 业务含义：体重差不多是历史的2倍，大概率是【录入时单位搞错：把斤当成kg录入】
print('是此前中位数1.8-2.2倍的条数：',
(raw['weight_kg']/prior).between(1.8,2.2).sum())

# 构造筛选条件：体重是历史中位数的1.8~2.2倍，并且本身数值<150
# 这类是典型：按斤录入，但数值又没到150，单纯靠>150的规则抓不到
jin = (raw['weight_kg']/prior).between(1.8,2.2) & (raw['weight_kg']<150)

# 取出第一条符合上面条件的居民ID，把这个人的全部随访记录打印出来，用来人工核查
raw[raw['resident_id']==raw.loc[jin,'resident_id'].iloc[0]]

>150kg的条数 97
是此前中位数1.8-2.2倍的条数： 335


,resident_id,visit_date,weight_kg
151,R0018839,2023-01-02,57.4
12939,R0018839,2023-05-10,118.8
35202,R0018839,2023-11-13,60.0
57903,R0018839,2024-05-23,59.5
77813,R0018839,2024-11-15,59.2
96286,R0018839,2025-05-11,59.8


### 复盘7：清洗规则本身也有时点

理解：清洗规则也必须遵守时间逻辑，不能拿未来的数据去修正过去的数据，这是非常隐蔽的数据泄露。

**总结**：

只靠 `>150` 这个简单阈值，只能抓到 **97 条**单位录错（斤写成 kg）的数据；
用历史体重倍数的时序规则，能抓到 **335 条**，很多错误记录体重本身不到 150，单阈值识别不到。
示例打印的这个人：57.4 → 118.8 → 59.4，118.8 刚好接近之前 57.4 的 2 倍，明显是录入时把斤当成 kg 了，正确值应该是 118.8 / 2 = 59.4kg。

清洗规则也必须遵守时间逻辑，不能拿未来的数据去修正过去的数据，这是非常隐蔽的数据泄露。
1. 业务背景：体重存在录入 bug，有时候把斤直接当成 kg 录入，数值翻倍；
2. 这里 `expanding()` 滚动窗口，只使用这条随访时间之前的历史体重，是正确写法；
3. 错误做法：直接拿这个人全部随访（包含未来）的体重中位数做参照。举例子：2023 年的那条体重要不要修正，如果参考值用到了 2025 年的体重，相当于建模时拿着未来信息清洗过去，属于时间泄露；模型上线时，2023 年的时候根本还没有 2025 年的数据，这套规则在真实线上无法使用。
4. 深层思想：任何清洗、校验规则，都要站在当时那个时刻去思考：这个信息在当时是否已经存在。清洗不是静态一次性操作，清洗规则自带时间维度。

**正确做法**：单位修正只用当天及以前的记录：与此前随访中位数之比在 1.8～2.2 之间判为"斤"；首次随访没有参考，用 BMI > 40 兜底。

In [29]:
# 从sklearn导入average_precision_score，用来计算AP（PR曲线下面积），适合不平衡样本评估
from sklearn.metrics import average_precision_score

# 使用的验证集
# 计算【全判阴性】这个无脑基线的准确率：全部预测为0（不患病）
print("全判阴性的准确率 %.3f" %(1 - y_valid.mean()))

# 循环遍历两组模型：逻辑回归、随机森林；分别传入预测标签pred和预测概率pro
for name, pred_, proba_ in [('逻辑回归', pred, pro), ('随机森林', pred_rf, pro_rf)]:
    # 输出模型名称、准确率、AP值
    print("%s: 准确率 %.3f AP %.4f" 
          % (name, accuracy_score(y_valid, pred_), average_precision_score(y_valid, proba_)))

# 业务场景：只有15%的复查名额，优先按风险分抓高危人群
# 公式：名额比例 / 患病率 = 最多能抓到的阳性样本上限（召回率上限
print("15% 名额能抓到的阳性上限 ",round(0.15 /y_valid.mean(),3))

全判阴性的准确率 0.701
逻辑回归: 准确率 0.956 AP 0.9506
随机森林: 准确率 0.960 AP 0.9533
15% 名额能抓到的阳性上限  0.502


### 复盘8：指标会骗人，分数要带口径

**总结**：

（1）全判阴性基线准确率 **0.701**：就算模型什么都不学，全部说所有人都不患病，也能拿到 70.1% 的准确率，这个就是准确率的底线。
（2）两个模型准确率看着很高（95%+），AP 也很高。
（3）15% 名额上限 **0.502**：如果只能选前 15% 高危人群复查，理论上最多抓到全部阳性里的 50.2%，不可能超过这个召回上限。

准确率是非常容易误导人的指标，在样本不平衡场景下失效：
- 这个数据集阳性样本少、阴性样本多。单纯看准确率看不出模型好坏：哪怕一个模型完全没有识别阳性的能力，全部预测阴性，也能拿到很高的准确率（这里 0.701）。
- AP（PR 曲线下面积）看重：在找阳性这件事上的排序能力，更适合筛查、患病预测这类不平衡任务。只看准确率，会误以为模型效果几乎没变；但 AP 反映出模型识别阳性的能力已经明显变差，准确率掩盖了模型真实的退化。

**结论**：保存模型指标的时候，不能只存数字，必须同时记录数据集口径：患病率、切分时间、清洗规则、样本范围。

In [30]:
# 选取 district(辖区)、education(教育程度)、income_level(收入等级)三列
# isna()标记是否缺失，mean()计算每一列缺失比例，保留3位小数，转为字典输出
print(df[['district','education','income_level']].isna().mean().round(3).to_dict())
# 按照辖区district分组，计算每组目标变量TARGET的均值（也就是每个辖区的患病率）
df.groupby('district')[TARGET].mean().round(3).to_dict()

{'district': 0.0, 'education': 0.0, 'income_level': 0.169}


{'D01': 0.306,
 'D02': 0.296,
 'D03': 0.309,
 'D04': 0.3,
 'D05': 0.292,
 'D06': 0.29,
 'D07': 0.308,
 'D08': 0.288}

### 复盘9：元数据被当成了特征

**总结**：

（1）缺失情况：`district`、`education` 完全没有缺失；`income_level` 缺失比例 **16.9%**。
（2）分组患病率：各个辖区患病率接近，大约在 **0.29 ~ 0.31** 之间。

**核心概念**：区分【观测特征】和【元数据】。
- **观测特征**：本次随访、体检直接测出来的指标（血压、血糖、体重等），描述这次随访测到了什么，用来预测患病风险。
- **元数据**：`district`、`education`、`income_level` 这类，描述这个人属于哪一个群体，属于人群属性标签，不是本次体检观测指标。

（1）元数据正确用法：**分组评估**
用辖区、收入、学历分组，看模型在不同人群上效果是否一致，检查模型公平性、有没有偏倚。比如：是不是 D01 区模型预测更准、低收入人群召回率偏低。

（2）**不建议直接放进模型当预测特征**
一旦把辖区、收入、学历喂给模型：同样的体检指标，仅仅因为住在不同辖区、收入不同，风险分就不一样。在医疗筛查分配复查名额的场景下，会带来解释性难题：两个人体检数据一模一样，只因为所在片区不同，一个优先复查、一个不优先，很难给业务方和患者解释，容易造成不公平。

In [31]:
# 拼接【居民ID + 竖线分隔 + 随访日期（只保留年月日）】，生成“某人某一天”的唯一标识key
# 用来识别同一个人同一天有多条随访记录
day_key = visits['resident_id'] + '|' + visits['visit_date'].dt.strftime('%Y-%m-%d')
# 直接用drop_duplicates()默认去重：整行完全一模一样才算重复
print('drop_duplicates()删除：',len(visits) - len(visits.drop_duplicates()),'行')

# 判断day_key是否重复，统计有多少组【同一个人同一天】的重复记录
print('同人同日的重复：',day_key.duplicated().sum(),'对')
# 筛选出所有属于“同人同日重复”的记录，并且按居民id、提交时间created_at排序
dup = visits[day_key.isin(day_key[day_key.duplicated()])].sort_values(['resident_id','created_at'])
# 只展示重复记录里关心的几个字段，输出前4条看样例
dup[['visit_id','resident_id','visit_date','created_at','weight_kg','sbp']].head(4)

drop_duplicates()删除： 0 行
同人同日的重复： 437 对


,visit_id,resident_id,visit_date,created_at,weight_kg,sbp
14580,V00014581,R0000065,2023-05-27,2023-05-27 13:05:59,62.6,126.0
14581,V00014582,R0000065,2023-05-27,2023-05-27 16:23:16,62.6,126.0
74502,V00074503,R0000153,2024-10-18,2024-10-18 08:37:17,72.8,121.0
74501,V00074502,R0000153,2024-10-18,2024-10-18 16:25:10,72.8,121.0


### 复盘10：重复提交

重复提交：同一个人同一天重复提交。

**总结**：

直接写 `drop_duplicates()` 是陷阱，它只能删掉【所有字段完全一模一样】的行，抓不到业务意义上的重复提交。

1. 业务场景：医生录入随访，不小心点两次提交，生成两条记录；体检指标一样、居民和日期一样，但是记录 ID、入库时间不一样。
2. 两种去重逻辑区别：
   - `drop_duplicates()`：技术层面，整行全部字段相等才判定重复；这里失效，删不掉。
   - 按业务主键（`resident_id + visit_date`）：业务层面，同一个人同一天的随访，就算入库时间不同，也属于重复提交，这才是我们要找的重复。
3. 风险：如果只简单写 `drop_duplicates`，会误以为已经去重完成，实际上重复样本还留在数据集里，同一个人的同一天数据多次进入训练集，造成样本重复、分布失真，模型评估虚高。

### 复盘11：问题的处理，查看指标的变化

In [32]:
# 定义函数：固定一套逻辑回归流水线，只修改数据口径，返回一轮评估结果
def score_once(data,num_cols,cat_cols,split='random'):
    # 判断切分方式：随机切分
    if split == 'random':
        tr,va = train_test_split(
            data,test_size=0.3, 
            random_state=SEED, 
            stratify=data[TARGET]
            )
    else:
        # 时间切分方式：预先已经标记好 _split 字段，train是历史(理解：2023/2024)、valid是未来(2025)
        tr,va = data[data['_split']=='train'], data[data['_split']=='valid']
    # 搭建流水线：先做预处理，再接入逻辑回归模型
    p = Pipeline([
        ('pre', make_pre(num_cols,cat_cols)),
        ('lr', LogisticRegression(max_iter=1000))
    ])
    # 使用数值特征+分类特征训练模型，拿到验证集上预测为阳性的概率
    s = p.fit(tr[num_cols+cat_cols], tr[TARGET]).predict_proba(va[num_cols+cat_cols])[:, 1]
    # 打包返回指标：训练样本量、验证样本量、验证集患病率、特征数量、AUC、AP
    result = {
        "训练": len(tr), "验证": len(va),"验证集患病率": round(va[TARGET].mean(), 3), 
        "列数": len(num_cols + cat_cols), 
        # 验证集中真实的患病率与预测的患病率
        "AUC": round(roc_auc_score(va[TARGET], s), 4), 
        "AP": round(average_precision_score(va[TARGET], s), 4)
    }
    return result

# 数值特征列表剔除 known_diabetes（确诊标签，属于数据泄露特征）
no_known = [c for c in NUM_COLS if c !='known_diabetes']
# 分类特征列表剔除 chronic_mgmt_tag（慢病管理标签，也是泄露特征）
no_tag = [c for c in CAT_COLS if c !='chronic_mgmt_tag']

# 时间切分：训练数据只取2024-03-02之前的随访记录
v_train = visits[visits['visit_date']<='2024-03-02']
# 验证数据：2024下半年随访，代表未来数据
v_valid = visits[visits['visit_date'].between('2024-07-01','2024-12-31')]
# 调用标签函数，给训练集、验证集打上_split标记，再和居民基础信息合并
pit = pd.concat([
    label_visits(v_train, pd.Timestamp("2024-06-30")).assign(_split="train"),
    label_visits(v_valid, pd.Timestamp("2025-06-30")).assign(_split="valid")
]).merge(residents,on='resident_id')

# 由出生日期和随访日期计算年龄
pit['age'] = (pit['visit_date']-pit['birth_date']).dt.days / 365.25
# 计算BMI：体重kg / 身高(m)的平方
pit['bmi'] = pit['weight_kg'] / (pit['height_cm']/100)**2
# 筛选：只保留未确诊糖尿病人群（符合筛查业务目标：找潜在新发病人）
pit = pit[pit['known_diabetes']==0]

# 字典保存5组不同口径的实验结果
rows = {
    # 原始版本v1：全人群、随机切分、全部特征
    "原始版本v1": score_once(df, NUM_COLS, CAT_COLS),
    
    # 第一刀：剔除已经确诊的人群，只筛未确诊人群，特征不变
    "拿掉已确诊的人群": score_once(df[df['known_diabetes'] == 0], no_known, CAT_COLS),
    
    # 第二刀：只剔除泄露标签chronic_mgmt_tag，人群不变
    "拿掉chronic_mgmt_tag": score_once(df, NUM_COLS, no_tag),
    
    # 两刀叠加：既剔除确诊人群，又剔除两个泄露特征
    "两个都拿掉": score_once(df[df['known_diabetes'] == 0], no_known, no_tag),

    # 第三刀：在前两刀基础上，再把随机切分改成时间切分（模拟真实上线：用历史训练、预测未来）
    "再换成时间点切分": score_once(pit, no_known, no_tag, split='time')
}

pd.DataFrame(rows).T

,训练,验证,验证集患病率,列数,AUC,AP
原始版本v1,55793.0,23912.0,0.299,25.0,0.9819,0.9506
拿掉已确诊的人群,45203.0,19374.0,0.143,24.0,0.9682,0.8945
拿掉chronic_mgmt_tag,55793.0,23912.0,0.299,24.0,0.9704,0.9330
两个都拿掉,45203.0,19374.0,0.143,23.0,0.9418,0.8123
再换成时间点切分,32762.0,13132.0,0.128,23.0,0.9317,0.7643


#### 解读：

1. 原始 v1 分数很高，是因为包含已经确诊的病人 + 泄露特征 + 随机切分，模型相当于开卷考试，分数虚高；
2. 第一刀拿掉确诊人群：任务变了，不再识别已经确诊的患者，而是筛查潜在新发患者，患病率直接腰斩，AUC、AP 明显下跌；
3. 第二刀拿掉泄露标签：去掉作弊信息，单独看 AUC 降幅不大，但和第一刀叠加之后 AP 大幅下滑；
4. 最后一刀换成时间切分：不再随机打乱，严格用历史数据训练、未来数据验证，最贴近上线真实场景，分数继续下降；
5. 整体变化：AUC 从 0.9819 掉到 0.9317，AP 从 0.9506 掉到 0.7643。全程模型、超参一点没改，仅仅是修改数据口径。

**核心问题**：高分很多时候不是模型强，是数据口径"作弊"带来的虚高，这就是 0.98 的高分消失的原因。
1. 随机切分的陷阱：随机切分把同一个人的历史、未来样本打散到训练集和验证集，模型提前看见了未来信息，评估结果乐观，不能代表上线效果；
2. 标签泄露：`known_diabetes`、`chronic_mgmt_tag` 是事后标签，属于泄露特征，模型直接靠这个标签猜结果，分数虚高；
3. 任务定义变化：原始数据集混着已经确诊的病人，和"筛查未确诊高危人群"的业务目标不一致；拿掉确诊样本之后，任务难度显著上升；
4. AUC 和 AP 的差异：AUC 对整体排序友好，不容易暴跌；AP 更关注高危名单靠前位置的召回效果，筛查业务里 AP 更贴近业务价值，口径改动时 AP 会暴跌，更能暴露模型的真实能力；
5. 实验思路：一刀一刀逐步修正口径，每一步都观察指标变化，定位高分是哪一步的"便利条件"带来的，而不是模型本身能力。

**总结**：模型不变，只改数据筛选范围、剔除泄露特征、把随机切分改为时间切分，指标就持续下跌；高 AUC/AP 是不严谨的实验口径带来的虚高，不是模型真实的上线能力。

## 7. 复盘2-工程问题

对的东西，能不能一直对下去？

### 1.train.parquet 是数据集吗？

问题：之前生成的 `train.parquet`，如果交给另一位同事去训练模型，他会怎么用？

讨论点：parquet 文件 ≠ 规范数据集，只存裸数据会带来下游失控风险。

In [33]:
# 导入依赖库
import pandas as pd
# 导入评估指标、流水线、逻辑回归，用于建模和计算AUC
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 读取训练集parquet文件
saved = pd.read_parquet('train.parquet')
# 查看保存的数据
print(saved.shape)
# 遍历全部列，找出不属于预先定义好的特征清单FEATURES的列（元数据、ID、时间、泄露字段等）
print("不是特征的列: ",[c for c in saved.columns if c not in FEATURES])
# 下游同事偷懒写法：去掉目标标签列，自动筛选所有数值类型的列当做模型输入特征
naive_cols = saved.drop(columns=[TARGET]).select_dtypes('number').columns.tolist()
# 搭建流水线：预处理函数 + 逻辑回归模型；这里下游直接用自动挑出来的数值列，没有遵守特征清单
naive = Pipeline([
    ('pre',make_pre(naive_cols,[])),
    ('lr',LogisticRegression(max_iter=2000))
])
# 用自动选出的数值特征训练模型
naive.fit(saved[naive_cols],saved[TARGET])
# 读取验证集valid.parquet，用来评估模型效果
held = pd.read_parquet('valid.parquet')
# 预测验证集上的阳性概率，计算ROC-AUC
auc_naive = roc_auc_score(held[TARGET],naive.predict_proba(held[naive_cols])[:,1])
# 打印这个偷懒自动选列方式得到的AUC（训练的 AUC: 0.9999）
print("下游直接 select_dtypes 训练的 AUC: %.4f" %auc_naive)

# 取出规定特征+目标标签，对分类列做独热编码get_dummies，生成宽表；
# 这是另一种常见错误做法：在数据集阶段就提前做编码，编码结果固化进数据集文件
wide = pd.get_dummies(df[FEATURES+[TARGET]],columns=CAT_COLS,dtype=int)
# 打印编码之后总列数，同时检查原始分类列（gender）是否还存在
print("先 get_dummies 再落盘: ",wide.shape[1],"列: 原始类别列还在吗? ","gender" in wide.columns)

(55793, 37)
不是特征的列:  ['visit_id', 'resident_id', 'visit_date', 'created_at', 'lab_id', 'sample_date', 'report_date', 'fpg_mmol_l', 'hba1c_pct', 'label', 'birth_date', 'enroll_date']
下游直接 select_dtypes 训练的 AUC: 0.9999
先 get_dummies 再落盘:  48 列: 原始类别列还在吗?  False


**问题**：

（1）上游导出 `train.parquet` 的时候，里面混杂了业务特征、ID、采样时间、检验时间、泄露字段。下游同事拿到文件，没有遵守约定的特征清单，直接 `select_dtypes("number")` 自动拿所有数字列训练，会偷偷把泄露列、元数据 ID、时间字段喂进模型。结果就是 AUC 虚高到 0.9999，看起来效果极好，但是上线之后完全不能用，因为线上推理拿不到这些泄露信息。根源：裸 parquet 文件没有契约，没有写死"哪些列允许进模型"，下游可以自由选列。

（2）很多人习惯提前用 `get_dummies` 把分类特征编码，编码之后的宽表直接存成 parquet。独热编码属于模型预处理逻辑，不是数据集本身。如果编码固化在数据集里：
- 以后要换模型、换编码方式，上游数据集文件就得全部重生成；
- 新样本出现训练集中没有的分类取值，推理直接报错；
- 数据集和预处理逻辑强绑定，流水线无法复用。

**工程对策**：

数据集是**语义宽表**：只保留原始语义字段，19 个原始输入特征 + 标签 + 6 个元数据，一共 26 列，列清单硬编码写死，作为上下游契约；

编码、标准化、缺失值填充这些预处理，全部放进 Pipeline，属于模型包，不属于数据集文件；

下游拿到 parquet，第一步先校验列：缺列、多列直接报错，不允许自动筛选特征，杜绝"随便挑数字列训练"。

为什么我们前面要求把预处理放进 Pipeline、固定特征清单、做列校验。不是为了跑分，是为了保证训练、验证、上线推理全程口径一致，今天实验是对的，上线之后、一个月之后迭代，依然是同一个口径，不会偷偷引入泄露、不会上下游理解不一致。

也就是标题问的：对的东西，能不能一直对下去？靠裸数据文件不行，要靠【数据集契约 + Pipeline 封装预处理】这套工程约束。

### 2.演示 A：训练和预测是两套代码

问题：同一批随访，离线用 `model.pkl` 打一次分，再像线上那样把原始记录逐条交给 `predict.py` 打一次分。两边一样吗？

In [34]:
# 从线上推理脚本predict.py导入predict推理函数，这就是线上服务跑预测的入口
# 【注意】predict.py 在项目根目录（上一级），notebook 的工作目录是 notebooks/，需要把根目录加入模块搜索路径
import sys
sys.path.append('..')
from predict import predict
import pandas as pd
import sqlite3

# 从验证集里随机抽样2000条随访记录
ids = valid.sample(2000,random_state=0)['visit_id']
# 【离线打分】
# 把验证集用visit_id设为索引，筛选出刚才抽样的2000条，使用训练时完整的PipeLine(best_pipe)、规定特征FEATURES计算预测概率
# predict_proba输出两列概率，[:,1]取正类风险概率；index=ids，保证结果索引和visit_id一一对应
offline = pd.Series(best_pipe.predict_proba(valid.set_index('visit_id').loc[ids,FEATURES])[:,1],index=ids)


# 打开sqlite数据库连接，线上真实场景是从数据库读取原始底层记录
with sqlite3.connect(DB) as conn:
    # 读取visits就诊表、residents居民表，按照resident_id关联合并，模拟线上收到的原始未加工数据
    online_raw = pd.read_sql("SELECT * FROM visits",conn).merge(
        pd.read_sql("SELECT * FROM residents",conn),on='resident_id'
    )
# 以visit_id设索引，只保留抽样的2000条；删掉不需要的ID、时间字段，交给线上predict函数
online_raw = online_raw.set_index('visit_id').loc[ids].drop(
    columns=["resident_id", "visit_date", "created_at", "birth_date", "enroll_date"]
)

# 【线上路径打分】
# 逐行遍历原始记录，先做dropna，转成字典，调用predict.py里的线上推理函数，生成线上版本风险分
# 最终生成Series，索引同样是visit_id，和离线分数对齐
online = pd.Series([predict(r.dropna().to_dict()) for _, r in online_raw.iterrows()],index=ids)

# 计算两组分数的绝对差值，用来衡量离线训练打分 和 线上服务打分 的偏差
diff = (online - offline).abs()
#diff

# 打印偏差统计指标：中位数偏差、90%分位偏差、最大偏差
print(f"中位数差 {diff.median():.4f} ,90% 分位差 {diff.quantile(0.9):.4f},最大差 {diff.max():.4f}")
# 统计偏差大于0.05的样本占比，看严重不一致的比例
print(f"差值超过0.05的比例 {(diff>0.05).mean():.1%}")

# 取前15%高风险样本数量k，业务上关注高风险名单是否一致（不是只看分数，还要看召回名单）
k = int(len(ids) * 0.15)
# 求离线前15%高风险visit_id集合 和 线上前15%集合的交集，计算重合率
print(f"前 15% 名单重合率 {len(set(offline.nlargest(k).index) & set(online.nlargest(k).index)) / k:.1%}")

中位数差 0.0100 ,90% 分位差 0.0450,最大差 0.2600
差值超过0.05的比例 8.4%
前 15% 名单重合率 84.0%


**总结**：

同一批样本，一套走训练时的离线 Pipeline 打分，一套走线上 `predict.py` 推理打分，分数不一样、高风险名单也对不上。
输出结果：中位数差 0.01，最大偏差 0.26，前 15% 高危名单重合率只有 84%，说明离线实验效果，不能原样搬到线上。

**偏差的根源**：
1. 派生特征逻辑分家：训练时会用 `visit_date - birth_date` 计算年龄，线上 `predict.py` 没有计算年龄，缺失值直接填充成训练集中位数 60 岁；BMI 等衍生字段也是两套独立代码，逻辑分叉。
2. 分类文本清洗不一致：训练时统一规范了"男、never"这类文本，线上没做归一，独热编码后全部变成 0。
3. 原始数据单位、哨兵值没有统一管控：身高、体重、异常 0 值，训练和线上处理规则不一样。
4. 最关键坑：虽然我们保存了 `model.pkl` 整条 Pipeline，特征派生、清洗逻辑写在 Pipeline 外面，没有跟着模型包一起序列化保存。

这类问题运行不会报错，属于**静默 bug**，不会抛出异常，但是预测结果悄悄跑偏。
结果：现在实验跑对了，上线之后就不对，没法一直对下去。

**工程对策**：

把【原始数据清洗、派生特征、预处理流水线、模型、业务阈值】打包成一个统一包 **ModelBundle**。训练环节、线上推理强制走完全相同的一套代码路径，不允许训练、预测分开写两套逻辑。并且增加**一致性测试用例**：固定一条输入样本，规定离线 Pipeline 和线上服务输出分数必须在允许误差范围内，作为上线门禁。

这个代码是一致性校验脚本，用来复现"训练代码和线上推理代码分离带来的线上线下不一致"这个经典工业机器学习坑；证明：只保存模型，不把全部特征加工逻辑封装进模型包，离线指标不可信，上线会翻车。

### 3.演示 B：random_state=42 就可复现了吗？

In [35]:
import pandas as pd

# 将就诊表visits和检验表labs基于resident_id居民id左连接，构造版本B的数据集df_b
df_b = visits.merge(labs, on='resident_id', how='left')

# 筛选时间窗口：检验样本日期 >= 就诊日期，并且检验样本日期在就诊日期之后90天以内
in_window_b = (df_b['sample_date'] >= df_b['visit_date']) & \
              (df_b['sample_date'] <= df_b['visit_date'] + pd.Timedelta(days=90))

# 只保留窗口内的检验记录；先按【样本日期、lab_id检验编号】排序，再按visit_id去重，保留第一条
df_b = df_b[in_window_b].sort_values(['sample_date', 'lab_id']).drop_duplicates('visit_id', keep='first')
# 构造标签TARGET：满足空腹血糖≥7.0 或者 糖化血红蛋白≥6.5 则标记为1，否则0，转为整数
df_b[TARGET] = ((df_b['fpg_mmol_l'] >= 7.0) | (df_b['hba1c_pct']>=6.5)).astype(int)

# 把居民基础信息表residents左连接进来，补充年龄、身高体重等基础字段
df_b = df_b.merge(residents,on='resident_id',how='left')
df_b['age'] = (df_b['visit_date'] - df_b['birth_date']).dt.days / 365.25
df_b['bmi'] = df_b['weight_kg'] / (df_b['height_cm']/100)**2

# 对比旧版本df和新版本df_b的标签：以visit_id为索引，按索引排序后，对比标签是否完全一致
same = df.set_index('visit_id')[TARGET].sort_index().equals(df_b.set_index('visit_id')[TARGET].sort_index())
print("两版标签完全一样：", same)

# 分别用旧df、新df_b跑同一个逻辑回归评估函数，打印两边得到的AUC指标
print(
    "逻辑回归 AUC:", 
    score_once(df, NUM_COLS, CAT_COLS)['AUC'], 
    '->', 
    score_once(df_b, NUM_COLS, CAT_COLS)['AUC']
    )


两版标签完全一样： True
逻辑回归 AUC: 0.9819 -> 0.9828


**总结**：

输出结果：两版标签完全一样: True，但是 AUC 从 0.9819 变成 0.9828。

标签一模一样，只是去重时的排序键多写了一个 `lab_id`，属于完全合理、业务上都成立的写法，最后模型指标却变了。

很多人误以为：只要设置 `random_state=固定种子`（比如 42），实验就能复现。但这个例子证明：`random_state` 只能保证你自己同一套代码，每次运行结果相同；它管不住上游数据清洗、去重、排序逻辑带来的不确定性。就算没有任何随机抽样、没有随机划分，仅仅是排序字段写得不一样，同样能跑出不一样的 AUC。这两个 AUC 数值没有谁对谁错，两种写法都符合业务规则，指标本身天然不稳定。

本质原因：同一个 `visit_id` 下，窗口内存在多条检验记录，选哪一条作为这条样本的代表，是由排序规则决定的。排序键写不全，就存在"歧义"。

**工程对策**：
1. 数据集划分优先用时间切分（按日期），而不是随机划分，规避随机带来的波动；
2. 数据去重、取首条这类操作，排序键必须写全，把所有决定先后顺序的字段都写上，保证取出来的记录是唯一确定的，消除歧义；
3. 给数据集生成内容指纹（hash），判断两份数据集是不是一模一样，不能靠肉眼看标签、看指标，用哈希值做精准比对。

这个案例用来打破学生的误区：固定随机种子 ≠ 实验可复现，数据预处理环节的隐性歧义（排序、去重、取记录）是机器学习实验不可复现的另一大来源。

呼应：对的东西，能不能一直对下去？只固定随机种子不够，要把上游数据加工规则写死、消除歧义，结果才稳定可复现。

### 4.全都不报错

In [36]:
# 打开sqlite数据库连接，自动管理连接，执行结束后自动关闭
with sqlite3.connect(DB) as conn:
    # 读取visits就诊表全部原始数据，保存到v0，作为清洗前基准版本
    v0 = pd.read_sql("select * from visits",conn)
    # 读取residents居民信息表全部原始数据，保存到r0，作为清洗前基准版本
    r0 = pd.read_sql("select * from residents",conn)

# 定义字典，用来统计每一条数据清洗规则，会影响/改动多少行数据
changed = {
    # 去重：原始总行数 - 去重之后行数 = 被删掉的重复记录数量
    "drop_duplicates()": len(v0) - len(v0.drop_duplicates()),
    # 性别写法异常：统计gender不在["M","F"]合法取值内的行数（非法编码、错别字、空值等）
    "性别写法": (~r0['gender'].isin(['M','F'])).sum(),
    # 吸烟状态异常：统计smoking_status不在规定合法枚举内的行数
    "吸烟写法": (~v0["smoking_status"].isin(["never", "former", "current"])).sum(),
    # 饮酒频率异常：统计drinking_freq不在规定合法枚举内的行数
    "饮酒写法": (~v0["drinking_freq"].isin(["never", "occasional", "frequent"])).sum(),
    # 身高异常：身高cm小于3，属于明显不合理脏数据
    "身高 < 3": (v0["height_cm"] < 3).sum(),
    # 体重异常：体重kg大于150，超出合理范围
    "体重 > 150": (v0["weight_kg"] > 150).sum(),
    # 收缩压异常：等于0，是哨兵缺失标记，不是真实测量值
    "收缩压 == 0": (v0["sbp"] == 0).sum(),
    # 每周运动时长异常：分钟数大于3000，超出合理范围
    "运动 > 3000": (v0["exercise_min_week"] > 3000).sum()
}

# 将字典转为Series，名字为改动行数，再转置（.T）成单行宽表
pd.Series(changed,name='改动行数').to_frame().T


,drop_duplicates(),性别写法,吸烟写法,饮酒写法,身高 < 3,体重 > 150,收缩压 == 0,运动 > 3000
改动行数,0,692,4165,3100,610,97,279,218


**解读**：

输出是一份清洗审计报表：
`drop_duplicates()`：0 行，说明原始数据没有重复；性别写法异常：692 行、吸烟写法异常：4165 行、饮酒写法异常：3100 行、身高异常：610 行、体重异常：97 行、收缩压 = 0：279 行、运动时长异常：218 行。

这些脏数据、格式错误、异常值，代码运行全程不会抛出任何报错，程序正常跑完，属于**静默故障**。比如：
- `drop_duplicates` 删 0 行，代码不报错；
- 体重 > 150 的 97 条脏数据，直接进模型，代码不报错；
- `reindex` 缺失字段自动填充空值、`dropna` 意外删掉大量男性样本，程序照样正常执行，不会提醒你。

Python、pandas 对这类脏数据高度宽容，不会主动告警，但是脏数据会悄悄污染模型，让实验结果不可信。

**论点**：

很多同学写数据清洗代码，只看"能不能跑通、不报错"，认为不报错 = 代码没问题。这个案例推翻这个认知：**不报错 ≠ 数据没问题**。脏数据、编码不一致、异常哨兵值、枚举不规范，全都可以安静跑完，埋在数据集里，最后导致模型效果异常、上线翻车。

**工程对策**：
1. 每一条清洗规则，都统计改动行数，输出一份清洗审计报告，每次跑实验都看：这次改了多少条、删除多少条、修正多少条，做到可追溯；
2. 把业务上的关键假设写成 assert 断言（校验守卫），一旦违反预设规则，直接抛出异常停止运行，而不是默默放过脏数据。

**总结**：这段代码是数据清洗审计脚本，用来统计每一类脏数据的数量，机器学习数据清洗不能只靠"看有没有报错"，必须主动监控每一步改动量，主动加校验守卫，提前发现静默脏数据问题。

**补充：assert 使用**

```python
assert 条件 , 报错信息
```

`assert` 的逻辑：条件为 True → 正常通过；条件为 False → 抛出 `AssertionError`，打印后面的提示文字。

示例：
```python
assert (~r0['gender'].isin(['M','F'])).sum() < 50, "性别异常样本太多，检查原始数据"
```